# Motor de consultas analíticas en Colab

Entorno de GPU: compila el backend CUDA, genera TPC-H SF10, mide serial / omp / cuda y grafica.
Antes de empezar, en el menú *Entorno de ejecución*, opción *Cambiar tipo de entorno*, elegir una GPU (T4, L4 o A100).

Clonar el repo (o, con `REPO_URL` vacío, subir un `.zip` del proyecto).

In [ ]:
# Ruta absoluta: al reejecutar la celda, el directorio actual ya es /content/proyecto.
%cd /content
REPO_URL = "https://github.com/LipezJ/motor-columnar-cpu-gpu.git"  # vacío: subir un .zip del proyecto
import os
if REPO_URL:
    !rm -rf proyecto && git clone -q {REPO_URL} proyecto
else:
    from google.colab import files
    zip_subido = next(iter(files.upload()))
    !unzip -q -o "{zip_subido}" -d proyecto_zip
    # El zip puede traer la carpeta del proyecto dentro o los archivos sueltos.
    raiz = next(os.path.dirname(r) for r, _, fs in os.walk("proyecto_zip") if "Makefile" in fs and "src" in os.listdir(r))
    !rm -rf proyecto && mv "{raiz}" proyecto
%cd /content/proyecto

In [ ]:
!pip install -q duckdb numpy matplotlib
!nvidia-smi

In [ ]:
import re, subprocess
consulta = lambda campo: subprocess.check_output(["nvidia-smi", f"--query-gpu={campo}", "--format=csv,noheader"], text=True).splitlines()[0].strip()
# compute_cap da la arquitectura directamente: 7.5 es sm_75 (T4), 8.0 es sm_80 (A100) y 8.9 es sm_89 (L4).
CUDA_ARCH = "sm_" + consulta("compute_cap").replace(".", "")
GPU = re.sub(r"^(NVIDIA|Tesla)\s+", "", consulta("name")).replace(" ", "-")
print(GPU, CUDA_ARCH)

In [ ]:
!make clean && make CUDA=1 CUDA_ARCH={CUDA_ARCH}

In [ ]:
# Validación rápida en SF1 antes de medir.
!test -d data/sf1 || python scripts/gen_datos.py 1
for q in ["q1", "q3", "q6"]:
    print(q, end=": ")
    !./build/motor data/sf1 cuda {q} --reps 3 | python scripts/validar.py data/sf1/esperado/{q}.csv
print("q1 ingenuo", end=": ")
!MOTOR_Q1_INGENUO=1 ./build/motor data/sf1 cuda q1 --reps 3 | python scripts/validar.py data/sf1/esperado/q1.csv

In [ ]:
# Si data/sf10 viene de una generación interrumpida con una versión anterior del script, se regenera.
!test -f data/sf10/esperado/q6.csv || (rm -rf data/sf10 && python scripts/gen_datos.py 10)

In [ ]:
!python scripts/bench.py --datos data/sf10 --backends serial,omp,cuda --etiqueta {GPU} --maquina {GPU}

In [ ]:
!python scripts/graficas.py resultados/*.csv
from glob import glob
from IPython.display import Image, display
for png in sorted(glob("resultados/graficas/*.png")):
    print(png)
    display(Image(png))

Descargar el CSV para versionarlo en `resultados/` del repo (junto a los de otras GPUs, `graficas.py` genera la comparación por GPU).

In [ ]:
from google.colab import files
files.download(f"resultados/{GPU}.csv")